# Translate Experiment Instructions + Build Priming Stimulus JSON

Two steps per language:
1. **Translate UI text** — translate `task_translation_template.xlsx` (instructions, consent, etc.) using the Facebook SeamlessM4T model and save `{lang}_experiment.csv` to `05_final_languages/{lang}/`.
2. **Build priming JSON** — read `{lang}_trials_final.csv` from `05_final_languages/{lang}/`, split into 8 randomised blocks, write per-block stimulus JSON files, and patch a copy of `spaml_template.json` with the language-specific file references, saving it as `{lang}_spaml.json`.

# Libraries, Models, and Functions

Load the SeamlessM4T model (used only for UI-text translation in step 1 — step 2 reads existing CSVs and needs no model).

In [1]:
import copy
import json
import math
import os

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoProcessor, AutoModelForSeq2SeqLM


# ---------------------------------------------------------------------------
# 1. Translate experiment UI text (instructions, consent form, etc.)
# ---------------------------------------------------------------------------

def translate_with_backtranslation(
    template_path,
    output_base_dir,
    target_lang="ukr",
    output_lang=None,
    source_lang="eng",
    batch_size=8,
):
    df = pd.read_excel(template_path)
    col = df.columns[0]
    texts = df[col].astype(str).str.strip().tolist()

    forward_translations = []
    back_translations = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Forward translation"):
        batch = texts[i : i + batch_size]
        inputs = processor(
            text=batch, src_lang=source_lang, return_tensors="pt", padding=True
        ).to(device)
        with torch.no_grad():
            output_tokens = model.generate(**inputs, tgt_lang=target_lang, max_new_tokens=256)
        forward_translations.extend(
            t.strip() for t in processor.batch_decode(output_tokens, skip_special_tokens=True)
        )

    for i in tqdm(range(0, len(forward_translations), batch_size), desc="Back translation"):
        batch = forward_translations[i : i + batch_size]
        inputs = processor(
            text=batch, src_lang=target_lang, return_tensors="pt", padding=True
        ).to(device)
        with torch.no_grad():
            output_tokens = model.generate(**inputs, tgt_lang=source_lang, max_new_tokens=256)
        back_translations.extend(
            t.strip() for t in processor.batch_decode(output_tokens, skip_special_tokens=True)
        )

    lang_code = output_lang if output_lang else target_lang
    output_dir = os.path.join(output_base_dir, lang_code)
    os.makedirs(output_dir, exist_ok=True)

    out_df = pd.DataFrame({
        "English": texts,
        "translation": forward_translations,
        "back_translation": back_translations,
    })
    output_path = os.path.join(output_dir, f"{lang_code}_experiment.csv")
    out_df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    return out_df


# ---------------------------------------------------------------------------
# 2. Build priming stimulus JSON files from trials_final.csv
# ---------------------------------------------------------------------------

# Path to the lab.js experiment template (relative to this notebook)
TEMPLATE_JSON_PATH = "../../03-Tasks/semantic_priming/spaml_template.json"

# Maps each block name to its component key inside spaml_template.json
BLOCK_COMPONENT_MAP = {
    "practice": "2",
    "real1":    "54",
    "real2":    "60",
    "real3":    "75",
    "real4":    "70",
    "real5":    "65",
    "real6":    "80",
    "real7":    "9",
    "real8":    "15",
}

N_REAL_BLOCKS = 8
N_PRACTICE    = 10


def _trials_to_stimulus_list(df, lang_code):
    """Convert rows of {lang}_trials_final.csv to lab.js templateParameters format."""
    cue_col    = f"{lang_code}_cue"
    target_col = f"{lang_code}_target"
    return [
        {
            "cue":   row[cue_col],
            "word":  row[target_col],
            "class": row["target_type"],  # 'word' or 'nonword'
            "type":  row["type"],         # 'related', 'unrelated', 'nonword'
        }
        for _, row in df.iterrows()
    ]


def build_priming_json(
    lang_code,
    output_base_dir,
    template_path=TEMPLATE_JSON_PATH,
    n_real_blocks=N_REAL_BLOCKS,
    n_practice=N_PRACTICE,
    random_state=42,
):
    """
    Reads {lang}_trials_final.csv, writes per-block stimulus JSON files,
    and saves a language-patched copy of spaml_template.json.
    """
    lang_dir    = os.path.join(output_base_dir, lang_code)
    trials_path = os.path.join(lang_dir, f"{lang_code}_trials_final.csv")

    if not os.path.exists(trials_path):
        print(f"[{lang_code}] No trials_final.csv found at {trials_path} — skipping.")
        return

    trials = (
        pd.read_csv(trials_path)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    # Separate practice and real trials
    practice_df = trials.head(n_practice)
    real_df     = trials.iloc[n_practice:].reset_index(drop=True)

    # Split real trials into equal blocks
    block_size = math.ceil(len(real_df) / n_real_blocks)
    blocks = {"practice": practice_df}
    for i in range(n_real_blocks):
        blocks[f"real{i+1}"] = real_df.iloc[i * block_size : (i + 1) * block_size]

    # Write per-block stimulus JSON files
    written = {}
    for block_name, block_df in blocks.items():
        stimulus_list = _trials_to_stimulus_list(block_df, lang_code)
        fname = f"stimuli_{lang_code}_{block_name}.json"
        fpath = os.path.join(lang_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            json.dump(stimulus_list, f, ensure_ascii=False, indent=2)
        written[block_name] = fname
        print(f"  [{lang_code}] {fname}: {len(stimulus_list)} trials")

    # Patch a deep copy of the template JSON
    with open(template_path, "r", encoding="utf-8") as f:
        template = json.load(f)

    patched = copy.deepcopy(template)

    for block_name, comp_key in BLOCK_COMPONENT_MAP.items():
        if block_name not in written:
            continue
        new_fname = written[block_name]
        comp      = patched["components"][comp_key]

        # Update the file reference entry
        for file_entry in comp.get("files", []):
            if isinstance(file_entry, dict) and "stimuli_english" in file_entry.get("localPath", ""):
                file_entry["localPath"] = new_fname
                file_entry["poolPath"]  = new_fname

        # Update the fetch call in the messageHandler
        for handler in comp.get("messageHandlers", []):
            if isinstance(handler, dict) and "code" in handler:
                old_name = f"stimuli_english_{block_name}.json"
                handler["code"] = handler["code"].replace(
                    f"this.files['{old_name}']",
                    f"this.files['{new_fname}']",
                )

    out_json_path = os.path.join(lang_dir, f"{lang_code}_spaml.json")
    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(patched, f, ensure_ascii=False, indent=2)

    print(f"\n[{lang_code}] Saved patched experiment JSON: {out_json_path}")

In [2]:
model_name = "facebook/seamless-m4t-v2-large"
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

SeamlessM4Tv2ForTextToText(
  (shared): Embedding(256102, 1024, padding_idx=0)
  (text_encoder): SeamlessM4Tv2Encoder(
    (embed_tokens): SeamlessM4Tv2ScaledWordEmbedding(256102, 1024, padding_idx=0)
    (embed_positions): SeamlessM4Tv2SinusoidalPositionalEmbedding()
    (layers): ModuleList(
      (0-23): 24 x SeamlessM4Tv2EncoderLayer(
        (self_attn): SeamlessM4Tv2Attention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (ffn): SeamlessM4Tv2FeedForwardNetwork(
          (fc1): Linear(in_features=1024, out_features=8192, bias=True)
          (fc2): Linear(in_features=8192,

## Step 1 — Translate UI text

Run once per language. Requires the model loaded above.

In [3]:
# translate into ukr
translate_with_backtranslation(
    template_path="task_translation_template.xlsx",
    output_base_dir="../05_final_languages",
    target_lang="ukr",
    output_lang="uk",
)

Back translation: 100%|██████████| 16/16 [03:23<00:00, 12.72s/it]

Saved: ../05_final_languages/uk/uk_experiment.csv


,English,translation,back_translation
0,(Block ${PAGE_INDEX + 1} of ${TOTAL_TASK_PAGES}),(Блок ${PAGE_INDEX + 1} з ${TOTAL_TASK_PAGES}),(This is a block of ${PAGE_INDEX+1} from ${TOT...
1,A copy of this information to keep for your re...,"Копія цієї інформації, яку ви повинні зберігат...","A copy of this information, which you must kee..."
2,Age,Вік,Age of the child
3,Age of Acquisition Study,Дослідження віку придбання,Age of acquisition research
4,Age ratings,Вікові рейтинги,Age ratings by age group
...,...,...,...
119,You've now completed the experiment.,Ви закінчили експеримент.,You've finished the experiment.
120,Your decision whether to participate will not ...,Ваше рішення про участь не вплине на ваші пото...,Your decision to participate will not affect y...
121,Your participant number is:,Ваш номер учасника:,Your participant number:
122,Your random participant id is:,Ваш ід випадкового учасника:,Your and the random participant:


## Step 2 — Build priming stimulus JSON

Reads `{lang}_trials_final.csv` from `05_final_languages/{lang}/`, shuffles trials, splits into 8 real blocks (625 trials each) plus a 10-trial practice block, writes per-block stimulus JSON files, and saves a language-patched `{lang}_spaml.json`.

No translation model needed — run this cell independently of step 1.

In [4]:
# Build priming JSON for a single language
build_priming_json(
    lang_code="uk",
    output_base_dir="../05_final_languages",
)

[uk] No trials_final.csv found at ../05_final_languages/uk/uk_trials_final.csv — skipping.


In [5]:
# Run build_priming_json for every language that has a trials_final.csv
base_dir = "../05_final_languages"

for lang in sorted(os.listdir(base_dir)):
    lang_dir = os.path.join(base_dir, lang)
    if not os.path.isdir(lang_dir):
        continue
    trials_path = os.path.join(lang_dir, f"{lang}_trials_final.csv")
    if os.path.exists(trials_path):
        print(f"\n=== {lang} ===")
        build_priming_json(lang_code=lang, output_base_dir=base_dir)


=== ar ===
  [ar] stimuli_ar_practice.json: 10 trials
  [ar] stimuli_ar_real1.json: 624 trials
  [ar] stimuli_ar_real2.json: 624 trials
  [ar] stimuli_ar_real3.json: 624 trials
  [ar] stimuli_ar_real4.json: 624 trials
  [ar] stimuli_ar_real5.json: 624 trials
  [ar] stimuli_ar_real6.json: 624 trials
  [ar] stimuli_ar_real7.json: 624 trials
  [ar] stimuli_ar_real8.json: 622 trials

[ar] Saved patched experiment JSON: ../05_final_languages/ar/ar_spaml.json

=== cs ===
  [cs] stimuli_cs_practice.json: 10 trials
  [cs] stimuli_cs_real1.json: 624 trials
  [cs] stimuli_cs_real2.json: 624 trials
  [cs] stimuli_cs_real3.json: 624 trials
  [cs] stimuli_cs_real4.json: 624 trials
  [cs] stimuli_cs_real5.json: 624 trials
  [cs] stimuli_cs_real6.json: 624 trials
  [cs] stimuli_cs_real7.json: 624 trials
  [cs] stimuli_cs_real8.json: 622 trials

[cs] Saved patched experiment JSON: ../05_final_languages/cs/cs_spaml.json

=== da ===
  [da] stimuli_da_practice.json: 10 trials
  [da] stimuli_da_real1.jso